In [3]:
!pip install huggingface transformers requests torch torchvision accelerate scikit-learn 
!pip install -U huggingface_hub 
!pip install IPython
import time 
from IPython.display import clear_output
print('all installs good so far') 
print('View the terminal output')
print("=" * 50)
time.sleep(10)
clear_output()

In [4]:
import os 
import torch 
import requests 
import transformers 
from PIL import Image 
import accelerate 

print(torch.cuda.is_available())
print(accelerate.__version__)

True
1.14.0


In [ ]:
# !hf auth login --token ""

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `woopwoop` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `woopwoop`


In [6]:
HOME = os.getcwd()
IMAGES_DIR = f'{HOME}/images'

if os.path.exists(IMAGES_DIR):
    dataset_length = len(os.listdir(IMAGES_DIR))
    print(f'{dataset_length} amount of images in dataset.')

2138 amount of images in dataset.


In [7]:
from transformers import Sam3Model, Sam3Processor 
device = "cuda:0"
model = Sam3Model.from_pretrained('facebook/sam3', device_map=device)
processor = Sam3Processor.from_pretrained('facebook/sam3')

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

In [8]:
'''SAM 3 AND YOLOE CAN BE USED FOR PASSING IN ONE-SHOT IMAGE EXAMPLES ALONG WITH TEXT PROMPTS FOR AUTO ANNOTATION'''

'SAM 3 AND YOLOE CAN BE USED FOR PASSING IN ONE-SHOT IMAGE EXAMPLES ALONG WITH TEXT PROMPTS FOR AUTO ANNOTATION'

In [9]:
images = os.listdir(IMAGES_DIR)
rgb_images = [Image.open(os.path.join(IMAGES_DIR, image)).convert('RGB') for image in images]
len(rgb_images)

2138

In [40]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [41]:
labels = ['deer', 'moose', 'coyote', 'package']
batch_size = 4
def multi_image_and_multi_label_inference(labels: list[str], rgb_images: list, all_results: list = []) -> dict: 
    batch_count = 0 
    for i  in range(0, len(rgb_images) - 2, batch_size): 
        start_time = time.time()
        batch_images = rgb_images[i: i + batch_size] # overflowing image index
        if len(batch_images) == 4: 
            print(type(batch_images))
            inputs = processor(images = batch_images, text = labels, return_tensors = 'pt').to(model.device)
            # text_inputs = processor(text = labels, return_tensors='pt').to(model.device)
            # img_inputs = processor(images=rgb_images, return_tensors='pt').to(model.device) - may be for image embeddings 
            with torch.no_grad():
                #text_embeds = model.get_text_features(**text_inputs)
                #vision_embeds = model.get_vision_features(pixel_values=img_inputs.pixel_values) 
                outputs = model(**inputs)
                #outputs = model(pixel_values = img_inputs.pixel_values, text_embeds = text_embeds, attention_mask=text_inputs.attention_mask)
            results = processor.post_process_instance_segmentation(outputs, threshold=0.5, mask_threshold=0.5, target_sizes=inputs.get('original_sizes').tolist())
            #results = processor.post_process_instance_segmentation(outputs, threshold=0.5, mask_threshold=0.5, target_sizes=img_inputs.get('original_sizes').tolist())[0]
                #all_results.append(results)
            #print(f'time took for one image inference {time.time() - start_time}')
            all_results.append(results)
            print(f'time took for one batch of 4 images to be inferred: {time.time() - start_time}')
            batch_count += 1
            #if torch.cuda.is_available(): torch.cuda.empty_cache()
        elif len(batch_images) < 4: 
            torch.cuda.empty_cache()
            inputs = processor(images = batch_images, text = labels, return_tensors='pt').to(model.device)
            with torch.no_grad():
                outputs = model(**inputs)
            results = processor.post_process_instance_segmentation(outputs, threshold=0.5, mask_threshold=0.5, target_sizes=inputs.get('original_sizes').tolist())
            all_results.append(results)
            batch_count += 1 
    print(f'total batches: {batch_count}')
    
    return all_results


In [42]:
start_time = time.time()
torch.cuda.empty_cache()
print(f'Starting multi image inference')
multi_image_and_multi_label_inference(labels, rgb_images)

print(f'took time: {time.time() - start_time} to run inference on {len(rgb_images)}')

Starting multi image inference
<class 'list'>
time took for one batch of 4 images to be inferred: 0.6498568058013916
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5532891750335693
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5484905242919922
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5730438232421875
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5537087917327881
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5465726852416992
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5884783267974854
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5503401756286621
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5527384281158447
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5972862243652344
<class 'list'>
time took for one batch of 4 images to be inferred: 0.5492818355560303
<class 'list'>
time too

In [ ]:
time took for one batch of 4 images to be inferred: 0.5516648292541504
total batches: 534
took time: 309.01989674568176 to run inference on 2138
took time: 382.1433584690094 to run inference on 2636 


# AUTO LABEL OUTPUT FILES 

# Necessary for YAML file: 
path: /absolute/path/to/your/yolo_dataset  # Root directory
train: images/train
val: images/val

# Class names
names:
  0: deer
  1: moose
  2: coyote
  3: package

# LABEL FILES - TXT 
empty file for no detections 
different text format for object detection/instance segmentation models 

# Object Detection Output (BBOX formats)
class_numbers center_x center_y width, height 
3 0.55 0.72 0.15 0.22 
0 0.12 0.45 0.30 0.50

# instance segmentation polygons 
class_id x1 y1 x2 y2 ... xn yn 
1 0.41 0.32 0.45 0.33 0.47 0.38 0.42 0.40 ...

# POST PROCESSING REQUIRED ON SAM3 -> outputs raw tensors (binary masks/unnormalized bounding boxes) -> have to calculate bboxes/
# extract polygons contours and normalize values against original image dimensions 

